In [9]:
import os
os.environ['XLA_PYTHON_CLIENT_PREALLOCATE'] = 'false'
from collections import namedtuple
import chex
from src.maenv.util import notify
import jax
import jax.numpy as jnp
from src.maenv.physics import Transform, RigidBody, CircleCollider


class UnitStatus(namedtuple('UnitStatus', ['id', 'health', 'attack_damage', 'attack_range', 'attack_cooldown', 'cooldown', 'sight_angle', 'sight_radius'])):
    id: chex.Array
    health: chex.Array
    attack_damage: chex.Array
    attack_range: chex.Array
    attack_cooldown: chex.Array
    cooldown: chex.Array
    sight_angle: chex.Array
    sight_radius: chex.Array

move_table = jnp.array(
        [
            [0, 1.0],
            [0, -1.0],
            [-1.0, 0],
            [1.0, 0],
            [0.0, 0.0],
            [0.0, 0.0],
        ]
    )


class UnitAction:
    UP = 0
    DOWN = 1
    LEFT = 2
    RIGHT = 3
    ATTACK = 4
    IDLE = 5

class DefaultUnit(namedtuple('DefaultUnit', ['transform', 'rigidbody', 'collider', 'team', 'pos_limit', 'status'])):
    transform: Transform
    rigidbody: RigidBody
    collider: CircleCollider
    team: chex.Array
    pos_limit: chex.Array
    status: UnitStatus
    
    def __new__(cls, transform, rigidbody, collider, team, pos_limit, status):
        return super().__new__(cls, transform, rigidbody, collider, team, pos_limit, status)
    
    def act(self, objects, action):
        # action : [rotate_angle, discrete action]
        is_attack = UnitAction.ATTACK == action[1]
        return notify(objects, 'hit', (action, self, is_attack))
    
    def on_hit(self, objects, info):
        attacker: DefaultUnit
        action, attacker, is_attack = info
        in_range = jnp.abs(attacker.transform.position - self.transform.position).sum() < attacker.status.attack_range #TODO : consider the angle
        is_enemy = attacker.team != self.team
        is_target = in_range & is_enemy & is_attack
        damaged_status = self.on_damage(attacker.status.attack_damage)
        new_status = jax.tree.map(lambda x, y: jnp.where(is_target, y, x), self.status, damaged_status)
        return self._replace(status=new_status)
        
    def on_damage(self, damage) -> UnitStatus:
        return self.status._replace(health=self.status.health - damage)
    
        

In [10]:
unit1 = DefaultUnit(
    transform=Transform(position=jnp.array([0.0, 0.0]), rotation=jnp.array([0.0])),
    rigidbody=RigidBody(mass=jnp.array([1.0]), velocity=jnp.array([0.0, 0.0]), acceleration=jnp.array([0.0, 0.0]), is_kinematic=jnp.array([False])),
    collider=CircleCollider(radius=1.0),
    team=jnp.array([0]),
    pos_limit=jnp.array([-10.0, 10.0]),
    status=UnitStatus(id=jnp.array([0]), health=jnp.array([100.0]), attack_damage=jnp.array([10.0]), attack_range=jnp.array([1.0]), attack_cooldown=jnp.array([0.0]), cooldown=jnp.array([0.0]), sight_angle=jnp.array([0.0]), sight_radius=jnp.array([10.0]))
)

unit2 = DefaultUnit(
    transform=Transform(position=jnp.array([0.0, 0.0]), rotation=jnp.array([0.0])),
    rigidbody=RigidBody(mass=jnp.array([1.0]), velocity=jnp.array([0.0, 0.0]), acceleration=jnp.array([0.0, 0.0]), is_kinematic=jnp.array([False])),
    collider=CircleCollider(radius=1.0),
    team=jnp.array([1]),
    pos_limit=jnp.array([-10.0, 10.0]),
    status=UnitStatus(id=jnp.array([1]), health=jnp.array([100.0]), attack_damage=jnp.array([10.0]), attack_range=jnp.array([1.0]), attack_cooldown=jnp.array([0.0]), cooldown=jnp.array([0.0]), sight_angle=jnp.array([0.0]), sight_radius=jnp.array([10.0]))
)

In [11]:
objecs = {
    'unit1' : unit1,
    'unit2' : unit2,
}

In [12]:
unit1.act(objecs, (jnp.array([0.0]), jnp.array([UnitAction.ATTACK])))

{'unit1': DefaultUnit(transform=Transform(position=Array([0., 0.], dtype=float32), rotation=Array([0.], dtype=float32)), rigidbody=RigidBody(mass=Array([1.], dtype=float32), velocity=Array([0., 0.], dtype=float32), acceleration=Array([0., 0.], dtype=float32), is_kinematic=Array([False], dtype=bool)), collider=CircleCollider(radius=1.0), team=Array([0], dtype=int32), pos_limit=Array([-10.,  10.], dtype=float32), status=UnitStatus(id=Array([0], dtype=int32), health=Array([100.], dtype=float32), attack_damage=Array([10.], dtype=float32), attack_range=Array([1.], dtype=float32), attack_cooldown=Array([0.], dtype=float32), cooldown=Array([0.], dtype=float32), sight_angle=Array([0.], dtype=float32), sight_radius=Array([10.], dtype=float32))),
 'unit2': DefaultUnit(transform=Transform(position=Array([0., 0.], dtype=float32), rotation=Array([0.], dtype=float32)), rigidbody=RigidBody(mass=Array([1.], dtype=float32), velocity=Array([0., 0.], dtype=float32), acceleration=Array([0., 0.], dtype=flo